In [ ]:
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install sqlalchemy
%pip install scikit-learn
%pip install python-dotenv
%pip install psycopg2-binary

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sqlalchemy import create_engine, URL


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 22.2 MB/s  0:00:00 eta 0:00:01


In [8]:
import psycopg2

print(psycopg2.__version__)

2.9.13 (dt dec pq3 ext lo64)


In [ ]:
# Get connection to DB

from pathlib import Path
from dotenv import load_dotenv

#Get env varibles from FetchData folder
env_path = Path("../../Fetch_Data/.env").resolve()
load_dotenv(env_path)

# Verify correct Path
print(env_path)
print(env_path.exists())

url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("POSTGRES_USER"),
    password=os.getenv("POSTGRES_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", "5433")),
    database=os.getenv("POSTGRES_DB")
)

engine = create_engine(url)

test_query = """
SELECT pokemon_name
FROM pokemon
"""

df = pd.read_sql(test_query, engine)
print(df.head())

/Users/brice/Documents/Projects/Pokemon_VGC_Analysis/Fetch_Data/.env
True
  pokemon_name
0    bulbasaur
1      ivysaur
2     venusaur
3   charmander
4   charmeleon


# Phase 1:

- Use logistical Regression to predict whether a team will make top cup (In this case defined as **top 10%**)
- First step consider a sub select of pokemon using one-hot encoding
    - Decide which pokemon to use via usage rates (only consider the most popular pokemon)
- Make different predictors per each regulation

## Example Encoding

| pk_1  | pk_2     | top_10%  |
|-------|----------|----------|
| 1     | 0        | 1        |
| 1     | 1        |0         |



In [ ]:
from sqlalchemy import text


sql_path = Path("../../analysis/top_10_per_reg.sql").resolve()

with open(sql_path, "r") as file:
    query = text(file.read())
    
regulation = "I"
num_pokemon = 10
min_event_size = 60




top_n_usage = pd.read_sql(
    query,
    engine,
    params={
        "regulation" : regulation,
        "num_pokemon" : num_pokemon,
        "min_event_size" : min_event_size
    }
)

top_names = top_n_usage["pokemon_name"].tolist()
print(top_names)


team_query = text("""
    SELECT
        t.team_id,
        t.placement,
        p.pokemon_name,
        e.event_id,
        e.event_size
    FROM teams t
    JOIN events e
        ON t.event_id = e.event_id
    JOIN team_pokemon tp
        ON t.team_id = tp.team_id
    JOIN pokemon p
        ON tp.pk_id = p.pk_id
    WHERE e.regulation = :regulation;
""")


team_pokemon = pd.read_sql(
    team_query,
    engine,
    params={
        "regulation" : regulation
    }
)


# Get all Team_id's from regulation
all_team_ids = team_pokemon["team_id"].unique()

# Get rid of non-top_n pokemon, because that data is not helpful this model
filtered = team_pokemon[team_pokemon["pokemon_name"].isin(top_names)]



# Use cross tab in order to get a count of pokemon per team id
ml_df = pd.crosstab(
    filtered["team_id"],
    filtered["pokemon_name"]
)

# Then get information about each team and where they placed
team_results = (
    team_pokemon[
        ["team_id", "placement", "event_id", "event_size"]
    ]
    .drop_duplicates()
    .set_index("team_id")
)

# Get teams that places in the top 10%
team_results["made_top_cut"] = (
    (
    team_results["placement"] / team_results["event_size"] <= 0.1
    ).astype(int)
)

# Then Ensure that everythin is in binary, in the rare case where it number > 1
ml_df = (ml_df > 0).astype(int)

# Reindex for teams that may not have any of the top N
# Also change df to desired columns 
ml_df = ml_df.reindex(
    index=all_team_ids,
    columns=top_names,
    fill_value=0
)

# Join this table with the made top 10 table
ml_df = ml_df.join(team_results["made_top_cut"])

# Seperate into X and Y data
X = ml_df.drop(columns=["made_top_cut"])
y = ml_df["made_top_cut"]


from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.2,
    random_state=42,
    stratify=y
)


model = LogisticRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
probabilities = model.predict_proba(X_test)


from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_pred
)


# Get results:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

accuracy = (tn + tp) / (tn + tp + fn + fp)
recall = tp / (tp + fn)
precison = tp / (tp + fp)

print("Accuracy: ",  accuracy)
print("Recall: ", recall)
print("Precision: ", precison)

from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["No Top Cut", "Top Cut"]
)

plt.title("Logistic Regression Confusion Matrix")
plt.show()


['incineroar', 'miraidon', 'calyrex-shadow', 'zamazenta', 'urshifu-rapid-strike', 'amoonguss', 'calyrex-ice', 'raging-bolt', 'rillaboom', 'chien-pao']


ValueError: y should be a 1d array, got an array of shape (814, 10) instead.